<a href="https://colab.research.google.com/github/lolo26130/media_restorer/blob/main/colab/server.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Media Restorer — Serveur Colab

**Utilisation :**
1. Exécuter **toutes** les cellules dans l'ordre (Runtime › Run all).
2. La dernière cellule affiche l'URL du tunnel — la coller dans
   Media Restorer › **Colab › Connecter Colab**.
3. Garder cet onglet Colab ouvert pendant l'utilisation.

Les librairies et les modèles sont sauvegardés sur votre Drive
(`/MyDrive/media_restorer_libs` et `/MyDrive/media_restorer_models`)
pour éviter les réinstallations à chaque session.

In [1]:
# ── Cellule 1 : montage Drive + installation des librairies ──────────
from google.colab import drive
drive.mount('/content/drive')

import sys, os

LIB_DIR   = '/content/drive/MyDrive/media_restorer_libs'
MODEL_DIR = '/content/drive/MyDrive/media_restorer_models'
os.makedirs(LIB_DIR,   exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

if LIB_DIR not in sys.path:
    sys.path.insert(0, LIB_DIR)

# Installe uniquement si le paquet est absent du Drive
def _need_install(pkg_name):
    return not any(
        os.path.isdir(os.path.join(LIB_DIR, d))
        for d in os.listdir(LIB_DIR)
        if d.lower().startswith(pkg_name.lower())
    )

pkgs = [
    ('realesrgan', 'realesrgan basicsr'),
    ('gfpgan',     'gfpgan facexlib'),
    ('fastapi',    'fastapi uvicorn python-multipart'),
]
for marker, install_str in pkgs:
    if _need_install(marker):
        print(f'Installation de {install_str}…')
        os.system(f'pip install {install_str} -t {LIB_DIR} -q')
    else:
        print(f'{marker} : déjà présent sur Drive.')

print('\nLibrairies prêtes.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Installation de realesrgan basicsr…
Installation de gfpgan facexlib…
Installation de fastapi uvicorn python-multipart…

Librairies prêtes.


In [3]:
# ── Cellule 2 : téléchargement des modèles (une seule fois) ──────────
import urllib.request
from pathlib import Path
MODEL_DIR = '/content/drive/MyDrive/media_restorer_models'
MODELS = {
    'Real-ESRGAN': (
        'RealESRGAN_x4plus.pth',
        'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth',
    ),
    'GFPGAN': (
        'GFPGANv1.4.pth',
        'https://github.com/TencentARC/GFPGAN/releases/download/v1.3.4/GFPGANv1.4.pth',
    ),
}

for engine, (fname, url) in MODELS.items():
    dest = Path(MODEL_DIR) / fname
    if dest.exists():
        print(f'{engine} : {fname} déjà présent.')
    else:
        print(f'{engine} : téléchargement de {fname}…')
        dest.parent.mkdir(parents=True, exist_ok=True) # Ensure parent directory exists
        urllib.request.urlretrieve(url, dest)
        print(f'  → {dest}')

print('\nModèles prêts.')

Real-ESRGAN : téléchargement de RealESRGAN_x4plus.pth…
  → /content/drive/MyDrive/media_restorer_models/RealESRGAN_x4plus.pth
GFPGAN : téléchargement de GFPGANv1.4.pth…
  → /content/drive/MyDrive/media_restorer_models/GFPGANv1.4.pth

Modèles prêts.


In [4]:
# ── Cellule 3 : chargement paresseux des moteurs ─────────────────────
import cv2
import numpy as np
from pathlib import Path

_engines = {}

def get_engine(name: str):
    """Retourne (et met en cache) le moteur demandé."""
    if name in _engines:
        return _engines[name]

    if name == 'Real-ESRGAN':
        from basicsr.archs.rrdbnet_arch import RRDBNet
        from realesrgan import RealESRGANer
        model = RRDBNet(
            num_in_ch=3, num_out_ch=3,
            num_feat=64, num_block=23, num_grow_ch=32, scale=4,
        )
        eng = RealESRGANer(
            scale=4,
            model_path=str(Path(MODEL_DIR) / 'RealESRGAN_x4plus.pth'),
            model=model,
            tile=512,
            half=True,
        )
        _engines[name] = eng
        return eng

    if name == 'GFPGAN':
        from gfpgan import GFPGANer
        eng = GFPGANer(
            model_path=str(Path(MODEL_DIR) / 'GFPGANv1.4.pth'),
            upscale=2,
            arch='clean',
            channel_multiplier=2,
        )
        _engines[name] = eng
        return eng

    raise ValueError(f'Moteur non supporté sur Colab : {name}')


def process_image(img_bgr: np.ndarray, engine_name: str) -> np.ndarray:
    """Applique le moteur et retourne l'image restaurée (BGR)."""
    eng = get_engine(engine_name)

    if engine_name == 'Real-ESRGAN':
        result, _ = eng.enhance(img_bgr, outscale=4)
        return result

    if engine_name == 'GFPGAN':
        _, _, result = eng.enhance(
            img_bgr,
            has_aligned=False,
            only_center_face=False,
            paste_back=True,
        )
        return result

    raise ValueError(f'process_image : moteur inconnu {engine_name}')


print('Moteurs définis (chargement différé au premier appel).')

Moteurs définis (chargement différé au premier appel).


In [5]:
# ── Cellule 4 : serveur FastAPI ───────────────────────────────────────
import base64, io, threading
import numpy as np
import cv2
import uvicorn
from fastapi import FastAPI, UploadFile, Form
from fastapi.responses import JSONResponse

app = FastAPI()

@app.get('/health')
async def health():
    return {'status': 'ok'}

@app.post('/process')
async def process(file: UploadFile, engine: str = Form('Real-ESRGAN')):
    data = await file.read()
    arr  = np.frombuffer(data, dtype=np.uint8)
    img  = cv2.imdecode(arr, cv2.IMREAD_UNCHANGED)
    if img is None:
        return JSONResponse({'error': 'Image illisible'}, status_code=400)
    try:
        result = process_image(img, engine)
    except ValueError as exc:
        return JSONResponse({'error': str(exc)}, status_code=422)
    ok, buf = cv2.imencode('.png', result)
    if not ok:
        return JSONResponse({'error': 'Encodage PNG échoué'}, status_code=500)
    return JSONResponse({'image': base64.b64encode(buf.tobytes()).decode()})

def _start():
    uvicorn.run(app, host='0.0.0.0', port=8000, log_level='warning')

threading.Thread(target=_start, daemon=True).start()
print('Serveur FastAPI démarré sur le port 8000.')

Serveur FastAPI démarré sur le port 8000.


In [6]:
# ── Cellule 5 : tunnel cloudflared → URL publique ────────────────────
import subprocess, re, time, os

CF = '/content/cloudflared'
if not os.path.exists(CF):
    os.system(
        'wget -q https://github.com/cloudflare/cloudflared/releases/latest'
        '/download/cloudflared-linux-amd64 -O /content/cloudflared'
        ' && chmod +x /content/cloudflared'
    )

proc = subprocess.Popen(
    [CF, 'tunnel', '--url', 'http://localhost:8000'],
    stderr=subprocess.PIPE,
    stdout=subprocess.DEVNULL,
)

url = None
deadline = time.time() + 20
while time.time() < deadline:
    line = proc.stderr.readline().decode(errors='replace')
    m = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', line)
    if m:
        url = m.group()
        break

if url:
    print('=' * 60)
    print('URL à coller dans Media Restorer → Colab → Connecter Colab :')
    print(f'  {url}')
    print('=' * 60)
else:
    print('Tunnel non trouvé dans le délai imparti — relancer la cellule.')

URL à coller dans Media Restorer → Colab → Connecter Colab :
  https://coordinate-spray-somewhat-aviation.trycloudflare.com
